# Block 1: setup + smoke + clean retrieval bằng model gốc, setup giống với bài

In [ ]:
%%bash
set -e

rm -rf /kaggle/working/usenix_1226_AE /kaggle/working/usenix_1226_AE_work_gte
cp -r /kaggle/input/datasets/pak2174/usenix-code/usenix_1226_AE /kaggle/working/
cd /kaggle/working/usenix_1226_AE

grep -v '^torch==' requirements_kaggle_core.txt > /tmp/requirements_kaggle_core_no_torch.txt
pip install -q -r /tmp/requirements_kaggle_core_no_torch.txt
pip install -q faiss-cpu

python - <<'PY'
from pathlib import Path

p = Path("generate_email.py")
s = p.read_text()

s = s.replace(
    "queries = [queries['top_10_questions'][i]['question'] for i in range(20)]",
    "queries = [item['question'] for item in queries['top_10_questions']]",
)

s = s.replace(
    "for query_idx in tqdm(range(10)):",
    "for query_idx in tqdm(range(min(10, len(queries)))):",
)

p.write_text(s)
print("Patched generate_email.py")
PY

python - <<'PY'
import torch
print("torch:", torch.__version__)
import faiss
print("faiss OK")
from sentence_transformers import SentenceTransformer
print("sentence_transformers OK")
PY

python main.py \
  dataset_path=/kaggle/working/usenix_1226_AE_work_gte \
  dataset_name=scifact \
  model_name=Alibaba-NLP/gte-modernbert-base \
  embedding_model=Alibaba-NLP/gte-modernbert-base \
  metric=dot \
  gpu_id=0 \
  attack_setting=clean \
  attack_tokenizer_name=gpt2 \
  our_adv_length=10 \
  our_num_samples_per_iter=5000 \
  our_elite_frac=0.2 \
  our_alpha=0.55 \
  our_num_iters=30 \
  position=prefix

# Block 2: poisoned retrieval, params giống bài

In [ ]:
%%bash
set -e
cd /kaggle/working/usenix_1226_AE

python main.py \
  dataset_path=/kaggle/working/usenix_1226_AE_work_gte \
  dataset_name=scifact \
  model_name=Alibaba-NLP/gte-modernbert-base \
  embedding_model=Alibaba-NLP/gte-modernbert-base \
  metric=dot \
  gpu_id=0 \
  attack_setting=poisoned \
  attack_tokenizer_name=gpt2 \
  our_adv_length=10 \
  our_num_samples_per_iter=5000 \
  our_elite_frac=0.2 \
  our_alpha=0.55 \
  our_num_iters=30 \
  position=prefix \
  num_query_tested=100 \
  '+suffix=This is an adversarial retrieval test document.'